<a href="https://colab.research.google.com/github/kovalchukolena99-droid/mlops-class-2026/blob/main/PAF_Project_Group_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================================
# PAF Project Group 6
# Analysis of XXX
# ==========================================================================

In [3]:
print("1. Upload the Highway accidents data")
from google.colab import files
uploaded = files.upload()

Upload the Highway accidents data


Saving highway_accidents.csv to highway_accidents (1).csv


In [8]:
import pandas as pd
import numpy as np
import requests
import geopandas as gpd
import warnings
from google.colab import files
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [11]:
df = pd.read_csv("highway_accidents.csv")
df.columns.tolist()

['date',
 'km',
 'road_direction',
 'roadway_type',
 'area_type',
 'latitude',
 'longitude',
 'time_of_day',
 'highway_type',
 'Uphill',
 'Crossing',
 'Curve',
 'Downhill',
 'Temporary_Detour',
 'Roadworks',
 'Road_Intersection',
 'Bridge',
 'Straight',
 'Regulated_Uturn',
 'Roundabout',
 'Tunnel',
 'Overpass',
 'accident_classification']

In [14]:
df["date"]=pd.to_datetime(df["date"])
df["year"]=df["date"].dt.year
df["month"]=df["date"].dt.month
df["day"]=df["date"].dt.day
df

,date,km,road_direction,roadway_type,area_type,latitude,longitude,time_of_day,highway_type,Uphill,...,Bridge,Straight,Regulated_Uturn,Roundabout,Tunnel,Overpass,accident_classification,year,month,day
0,2007-01-01,45.5,1.0,0.0,0.0,NaN,NaN,0,3.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2007,1,1
1,2007-01-01,83.9,1.0,0.0,1.0,NaN,NaN,1,3.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2007,1,1
2,2007-01-01,11.0,0.0,0.0,0.0,NaN,NaN,3,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2007,1,1
3,2007-01-01,180.0,1.0,2.0,1.0,NaN,NaN,0,4.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2007,1,1
4,2007-01-01,230.6,0.0,2.0,0.0,NaN,NaN,0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,2007,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,2025-12-31,412.5,0.0,0.0,1.0,-29.810654,-51.496686,3,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2025,12,31
149996,2025-12-31,51.0,1.0,0.0,0.0,-8.137602,-35.333387,2,4.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2025,12,31
149997,2025-12-31,645.5,1.0,0.0,0.0,-25.829749,-49.116659,0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2025,12,31
149998,2025-12-31,125.7,1.0,2.0,0.0,-23.969566,-50.235536,1,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,2025,12,31


In [26]:
print("2. Check for missing values")
missing = df.isnull().mean()*100
missing.round(2).sort_values(ascending=False)

2. Check for missing values


,0
longitude,70.82
latitude,70.82
road_direction,0.06
date,0.00
km,0.00
roadway_type,0.00
area_type,0.00
time_of_day,0.00
highway_type,0.00
Uphill,0.00


In [29]:
#From the first look we can see that around 70.82% of data in longitude and latitude
#are missing, while the rest of the fields have very little or no missing values.
# Next step is to analyse the reson why this data is missing and whether there is
# a particular pattern.

In [30]:
pd.crosstab(df["year"], df["latitude"].isnull(), normalize="index")*100

latitude,False,True
year,,
2007,0.0,100.0
2008,0.0,100.0
2009,0.0,100.0
2010,0.0,100.0
2011,0.0,100.0
2012,0.0,100.0
2013,0.0,100.0
2014,0.0,100.0
2015,0.0,100.0


In [ ]:
# Further analysis uncovers that the GPS data is missing for 2007-2016.
# This is an important boundary that also shows that the GPS recordings only
# started from 2016.


In [31]:
# Check prevalence of road infrastructure features
infra_cols = [
    'Straight',
    'Curve',
    'Crossing',
    'Downhill',
    'Uphill',
    'Road_Intersection',
    'Roundabout',
    'Regulated_Uturn',
    'Roadworks',
    'Overpass',
    'Bridge',
    'Temporary_Detour',
    'Tunnel',
]

infra_summary = pd.DataFrame({
    'Total Crashes': df[infra_cols].sum().astype(int),
    'Prevalence (%)': (df[infra_cols].mean() * 100).round(2),
}).sort_values(by='Total Crashes', ascending=False)

infra_summary

,Total Crashes,Prevalence (%)
Straight,105958,70.64
Curve,31319,20.88
Crossing,7098,4.73
Downhill,3481,2.32
Uphill,2694,1.80
Road_Intersection,2625,1.75
Roundabout,1009,0.67
Regulated_Uturn,858,0.57
Roadworks,677,0.45
Overpass,556,0.37
